# Generational Differences in Sharing Visuals by Phone and Preserving Digital Files

The term **visuals** refers to photos and videos. Choose an input mode:

- **`aggregate`** checks the supplied statistical tables and redraws nine figures. No survey file is needed.
- **`survey`** reads your selected SAV file, calculates new results, and draws the figures from those results.

Obtain official survey data through the [provider's access channels](../docs/DATA_ACCESS.md). The repository contains no respondent records. The [Notebook guide](../docs/NOTEBOOK.md) explains upload, configuration, local execution and downloads.

## 1. Prepare the input

For an aggregate preview, skip directly to the settings cell.

For survey analysis, obtain the official SAV and prepare its mapping profile as described in the [input-profile guide](../docs/INPUT_PROFILE.md). The profile connects the release's variables and response codes to the analysis; it does not contain respondent records.

**On Binder:** open the `data/official` folder in JupyterLab's left file browser, then click **Upload Files** to upload your SAV and profile. If your filenames are `survey.sav` and `profile.json`, their paths are:

```python
DATA_FILE = "/home/jovyan/data/official/survey.sav"
PROFILE_FILE = "/home/jovyan/data/official/profile.json"
```

Use Binder only for data you are permitted to process on this cloud service. Otherwise run the same Notebook locally. Uploads belong to the current session; this Notebook does not commit them to GitHub or include them in the results ZIP.

## 2. Edit the settings

Use `MODE = "aggregate"` for the supplied results, or `MODE = "survey"` for your SAV. Replace the example paths with your filenames. `CALIBRATION_FILE` is optional: it supplies population targets for the additional recalibration check; without it, that check is marked unavailable. Other checks use the supplied survey inputs.

Choose a new output folder for every run. Binder example: `/tmp/tscs-results/run-01`. Local example: `~/Documents/TSCS/results/run-01`. Leave `OUTPUT_DIR` blank for a fresh temporary folder. `FONT` can stay blank on Binder.

Press **Shift + Enter**, then run the remaining cells in order. After changing a setting, rerun from this cell onward.

In [ ]:
# Choose "aggregate" for supplied results or "survey" for your selected SAV.
MODE = "aggregate"
DATA_FILE = ""         # Binder: "/home/jovyan/data/official/survey.sav"
PROFILE_FILE = ""      # Binder: "/home/jovyan/data/official/profile.json"
CALIBRATION_FILE = ""  # Optional population-target JSON; blank skips that check.
OUTPUT_DIR = ""        # Binder: "/tmp/tscs-results/run-01"
FONT = ""              # Blank uses the installed default.

## 3. Confirm the settings

The next cell shows the selected mode and file paths. With `survey`, confirm these are the files you just uploaded. Local paths can start with `~`, which means your home directory.

Command-line execution replaces this settings cell only in the executed copy; the source Notebook stays unchanged.

In [ ]:
import sys
from pathlib import Path

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "run.py").is_file() and (candidate / "manifest.json").is_file():
        sys.path.insert(0, str(candidate))
        break

from helpers.notebook_workflow import configuration, find_root, run_workflow

root = find_root()
settings = configuration({
    "REPRO_MODE": MODE,
    "REPRO_DATA_FILE": DATA_FILE,
    "REPRO_PROFILE_FILE": PROFILE_FILE,
    "REPRO_CALIBRATION_FILE": CALIBRATION_FILE,
    "REPRO_OUTPUT_DIR": OUTPUT_DIR,
    "REPRO_FONT": FONT,
})
print("Mode:", settings.mode)
if settings.mode == "survey":
    print("Survey file:", settings.data_file)
    print("Mapping profile:", settings.profile_file)
    print("Population targets:", settings.calibration_file or "Not supplied")
else:
    print("Input: published aggregate tables included in this repository")
print("Output directory:", settings.output_dir)

## 4. Run the analysis and draw the figures

In `aggregate` mode, this step verifies the supplied tables, multiple-testing adjustments and sensitivity summaries.

In `survey` mode, it reads your SAV, applies the mapping profile, estimates the group results and runs the specified statistical comparisons and sensitivity checks. This can take several minutes. Missing required inputs stop the run; stored results are never substituted for a missing calculation.

Both modes produce nine figures and links to a preview and results ZIP.

In [ ]:
report = run_workflow(root, settings)
if report["statistical_reestimation"]:
    print("Calculated results from the selected survey file.")
    if report.get("artificial_data"):
        print("ARTIFICIAL DEMONSTRATION DATA: these are not empirical study results.")
    if not report["calibration_available"]:
        print("Additional population recalibration was not run: no targets were supplied.")
else:
    print("Verified published aggregates; survey statistics were not reestimated.")
print("Research figures:", report["figure_count"])

import json
from helpers.notebook_exports import export_results
from helpers.notebook_display import display_results

exports = export_results(root, settings.output_dir.resolve(), root / "notebooks/downloads")
download_report = {key: str(exports[key]) for key in ("directory", "index_path", "zip_path")}
(settings.output_dir / "download_report.json").write_text(
    json.dumps(download_report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)
print("Results directory:", settings.output_dir)
display_results(exports)

## 5. Inspect the figures

The figures compare weighted proportions within age groups, education groups, and groups within digitally mature or non-mature areas. The source mode is shown below. The plotting rows are saved in `figures/selected_profiles.csv`.

See the [terminology guide](../docs/TERMINOLOGY.md) for activity definitions, denominators and storage categories.

In [ ]:
from IPython.display import Image, Markdown, display

source = ("Reestimated from the selected survey file" if report["statistical_reestimation"]
          else "Published aggregates; not reestimated")
if report.get("artificial_data"):
    source = "Artificial demonstration data; not empirical study results"
display(Markdown(f"**Figure data source: {source}.**"))
for relative in report["figure_files"]:
    display(Image(filename=str(settings.output_dir / relative), width=1000))

## 6. Download the results

Use **View nine figures** or **Download figures and aggregates (ZIP)** above. The ZIP contains PNG/PDF figures, plotting aggregates, ten statistical result tables, and reading information. In `survey` mode, these are the newly calculated tables. It excludes the SAV, mapping profile, respondent records and execution logs.

Download copies are created under `notebooks/downloads/`, where Jupyter serves the links; they are ignored by Git. After extracting the ZIP, open `index.html` to view the figures. The tables are in `tables/primary/`, `tables/alternative/` and `tables/contrasts/`.

Binder has temporary storage. Download the results before ending the session. Clear execution outputs and reset personal paths before contributing a Notebook change.